# Exploration et analyse du catalogue Netflix.

In [62]:
import numpy as np
import pandas as pd

In [63]:
netflix = pd.read_csv('../data/raw/netflix_titles_raw.csv')

## Exploration du dataset

- Le nombre de colonnes
- Les noms de colonnes
- Le compte des valeurs manquantes
- Les dtypes par colonnes
- La taille du jeu de données
- Le nombre de cellules

### Informations sur le dataset

In [64]:
netflix.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


### Description des statistiques courantes du dataset

In [65]:
netflix.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


### La taille, le nombre de cellules et les dtypes du dataset

In [66]:
netflix.shape

(8807, 12)

In [67]:
netflix.size

105684

In [68]:
netflix.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

### Les valeurs manquantes

- Détecter les valeurs manquantes
- Compter le nombre de valeurs manquantes
- Calculer le pourcentage de valeurs manquantes

In [69]:
netflix.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [70]:
netflix.isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.299080
cast            0.093675
country         0.094357
date_added      0.001135
release_year    0.000000
rating          0.000454
duration        0.000341
listed_in       0.000000
description     0.000000
dtype: float64

#### Hypothèse

Est-ce que `director` est vide parce que la donnée n'a jamais été collectée (MCAR — manquant complètement au hasard), ou parce qu'il existe une vraie raison structurelle (par exemple : les TV Shows n'ont souvent pas de "réalisateur" unique désigné, contrairement aux films) ?

On distingue généralement trois types de données manquantes :

- MCAR (Missing Completely At Random) : la probabilité d'absence n'a aucun lien avec les données elles-mêmes ou d'autres variables (Rare en pratique).

- MAR (Missing At Random) : l'absence est liée à une autre colonne observable (ex : director manquant plus souvent quand type == "TV Show").

- MNAR (Missing Not At Random) : l'absence est liée à la valeur elle-même, non observable (ex : les gens à très haut revenu qui refusent de déclarer leur salaire).

> Pourquoi ça compte concrètement : si tu supprimes ou imputes sans comprendre le mécanisme, tu peux introduire un biais silencieux dans tes données. Si `director` manque majoritairement pour les séries TV, et que tu drop toutes les lignes avec director manquant, tu viens de supprimer presque toutes tes séries TV du dataset — sans le vouloir.

In [71]:
netflix[netflix['type'] == 'TV Show'].isna().sum()

show_id            0
type               0
title              0
director        2446
cast             350
country          391
date_added        10
release_year       0
rating             2
duration           0
listed_in          0
description        0
dtype: int64

In [72]:
netflix[netflix['type'] == 'TV Show'].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.914051
cast            0.130792
country         0.146114
date_added      0.003737
release_year    0.000000
rating          0.000747
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

##### Pour compléter la vérification de l'hypothèse

Taux conditionnel P(manquant | TV Show) vs Proportion P(TV Show | manquant)).

Avant la conclusion définitivement, faire le même calcul côté Movie sur director uniquement. Si le taux est très bas pour Movie et très haut pour TV Show, alors l'hypothèse MAR est solidement confirmée — c'est le contraste entre les deux groupes qui fait la preuve, pas juste un chiffre isolé.

In [73]:
netflix[netflix['type'] == 'Movie'].isna().sum()

show_id           0
type              0
title             0
director        188
cast            475
country         440
date_added        0
release_year      0
rating            2
duration          3
listed_in         0
description       0
dtype: int64

In [74]:
netflix[netflix['type'] == 'Movie'].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.030664
cast            0.077475
country         0.071766
date_added      0.000000
release_year    0.000000
rating          0.000326
duration        0.000489
listed_in       0.000000
description     0.000000
dtype: float64

Sur le contraste director — 3,07% pour Movie vs 91,4% pour TV Show. Ça, c'est une preuve solide et propre d'un mécanisme MAR : l'absence de director est quasi exclusivement structurelle (les séries n'ont généralement pas de réalisateur unique crédité), pas aléatoire. Tu peux documenter cette conclusion telle quelle dans ton rapport final, avec les deux chiffres côte à côte — c'est exactement le type de justification qu'on attend.

#### Hypothèse

Est ce que `cast` est vide parce que les documentaires et docuseries n'ont pas de casting ?


In [75]:
netflix[netflix['listed_in'].str.contains('Documentaries')].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.065593
cast            0.487917
country         0.086306
date_added      0.000000
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [76]:
netflix[netflix['listed_in'].str.contains('Documentaries')].isna().sum()

show_id           0
type              0
title             0
director         57
cast            424
country          75
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

In [77]:
netflix[netflix['listed_in'].str.contains('Docuseries')].isna().mean()

show_id         0.000000
type            0.000000
title           0.000000
director        0.848101
cast            0.524051
country         0.164557
date_added      0.002532
release_year    0.000000
rating          0.000000
duration        0.000000
listed_in       0.000000
description     0.000000
dtype: float64

In [78]:
netflix[netflix['listed_in'].str.contains('Docuseries')].isna().sum()

show_id           0
type              0
title             0
director        335
cast            207
country          65
date_added        1
release_year      0
rating            0
duration          0
listed_in         0
description       0
dtype: int64

Si ce nombre est 0, ton addition simple était correcte. S'il est différent de 0, il faut le soustraire de ta somme pour ne pas compter ces lignes deux fois (principe d'inclusion-exclusion : `|A ∪ B| = |A| + |B| - |A ∩ B|`).

In [79]:
424 + 207 + 0

631

##### Pour compléter la vérification de l'hypothèse

Hypothèse MAR confirmée à 76%, le reste (24%) reste sans explication claire pour l'instant, ce qui est normal

In [80]:
netflix[
    netflix['listed_in'].str.contains('Documentaries') &
    netflix['listed_in'].str.contains('Docuseries')
].shape[0]

0

In [81]:
631/825

0.7648484848484849

In [82]:
netflix[netflix['duration'].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


Tableau `duration` manquant (3 lignes) : les colonnes `rating` contiennent `"74 min", "84 min", "66 min"` — ce sont clairement des durées, pas des ratings (`P`G-13, TV-MA...). Et les 3 lignes sont toutes des specials de `Louis C.K.` Ce n'est pas un hasard : on peut parier que dans le CSV source, le champ rating était vide pour ces lignes précises (peut-être parce que ces contenus n'ont jamais reçu de classification officielle), et un bug de génération du dataset a décalé la valeur de duration vers la gauche pour combler le vide — laissant duration vide à la fin. Une vraie corruption structurelle, localisée à 3 lignes.

In [83]:
mask = netflix['duration'].isna()
netflix.loc[mask, 'duration'] = netflix.loc[mask, 'rating']
netflix.loc[mask, 'rating'] = np.nan
netflix.loc[mask]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,NaN,74 min,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,NaN,84 min,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,NaN,66 min,Movies,The comic puts his trademark hilarious/thought...


In [84]:
netflix[netflix['rating'].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,NaN,74 min,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,NaN,84 min,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,NaN,66 min,Movies,The comic puts his trademark hilarious/thought...
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava ...,NaN,"Oprah Winfrey, Ava DuVernay",NaN,"January 26, 2017",2017,NaN,37 min,Movies,Oprah Winfrey sits down with director Ava DuVe...
6827,s6828,TV Show,Gargantia on the Verdurous Planet,NaN,"Kaito Ishikawa, Hisako Kanemoto, Ai Kayano, Ka...",Japan,"December 1, 2016",2013,NaN,1 Season,"Anime Series, International TV Shows","After falling through a wormhole, a space-dwel..."
7312,s7313,TV Show,Little Lunch,NaN,"Flynn Curry, Olivia Deeble, Madison Lu, Oisín ...",Australia,"February 1, 2018",2015,NaN,1 Season,"Kids' TV, TV Comedies","Adopting a child's perspective, this show take..."
7537,s7538,Movie,My Honor Was Loyalty,Alessandro Pepe,"Leone Frisa, Paolo Vaccarino, Francesco Miglio...",Italy,"March 1, 2017",2015,NaN,115 min,Dramas,"Amid the chaos and horror of World War II, a c..."


Tableau `rating` manquant (4 lignes) : regarde bien — ici duration contient des valeurs parfaitement normales (`37 min, 1 Season, 115 min...`). Il n'y a aucun décalage. C'est juste rating qui est réellement absent, sans lien avec le bug repéré dans l'autre tableau. Deux causes différentes, deux corrections différentes.

#### Suppression vs Imputation (Remplacement des valeurs manquantes avec des valeurs explicites)

> Ne jamais supprimer une ligne entière à cause d'une seule colonne problématique quand le reste de la ligne a de la valeur. La suppression (dropna) ne devrait être qu'un dernier recours, pas un réflexe.

Petite nuance à ajouter pour les justifications futures (dans le rapport de nettoyage) : le choix "Unknown" plutôt que dropna() est d'autant plus justifié ici que le volume est négligeable (7/8807 ≈ 0,08%) — même si l'hypothèse d'imputation était légèrement imparfaite, l'impact sur une analyse globale serait quasi nul. C'est un raisonnement de risque : plus le volume concerné est petit, moins il faut être rigoureux sur le choix d'imputation (l'erreur potentielle est diluée) ; plus il est grand (comme director à 30%), plus la stratégie doit être réfléchie car une erreur là aurait un vrai impact statistique.

In [85]:
netflix[netflix['date_added'].isna()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
6066,s6067,TV Show,A Young Doctor's Notebook and Other Stories,NaN,"Daniel Radcliffe, Jon Hamm, Adam Godley, Chris...",United Kingdom,NaN,2013,TV-MA,2 Seasons,"British TV Shows, TV Comedies, TV Dramas","Set during the Russian Revolution, this comic ..."
6174,s6175,TV Show,Anthony Bourdain: Parts Unknown,NaN,Anthony Bourdain,United States,NaN,2018,TV-PG,5 Seasons,Docuseries,This CNN original series has chef Anthony Bour...
6795,s6796,TV Show,Frasier,NaN,"Kelsey Grammer, Jane Leeves, David Hyde Pierce...",United States,NaN,2003,TV-PG,11 Seasons,"Classic & Cult TV, TV Comedies",Frasier Crane is a snooty but lovable Seattle ...
6806,s6807,TV Show,Friends,NaN,"Jennifer Aniston, Courteney Cox, Lisa Kudrow, ...",United States,NaN,2003,TV-14,10 Seasons,"Classic & Cult TV, TV Comedies",This hit sitcom follows the merry misadventure...
6901,s6902,TV Show,Gunslinger Girl,NaN,"Yuuka Nanri, Kanako Mitsuhashi, Eri Sendai, Am...",Japan,NaN,2008,TV-14,2 Seasons,"Anime Series, Crime TV Shows","On the surface, the Social Welfare Agency appe..."
7196,s7197,TV Show,Kikoriki,NaN,Igor Dmitriev,NaN,NaN,2010,TV-Y,2 Seasons,Kids' TV,A wacky rabbit and his gang of animal pals hav...
7254,s7255,TV Show,La Familia P. Luche,NaN,"Eugenio Derbez, Consuelo Duval, Luis Manuel Áv...",United States,NaN,2012,TV-14,3 Seasons,"International TV Shows, Spanish-Language TV Sh...","This irreverent sitcom featues Ludovico, Feder..."
7406,s7407,TV Show,Maron,NaN,"Marc Maron, Judd Hirsch, Josh Brener, Nora Zeh...",United States,NaN,2016,TV-MA,4 Seasons,TV Comedies,"Marc Maron stars as Marc Maron, who interviews..."
7847,s7848,TV Show,Red vs. Blue,NaN,"Burnie Burns, Jason Saldaña, Gustavo Sorola, G...",United States,NaN,2015,NR,13 Seasons,"TV Action & Adventure, TV Comedies, TV Sci-Fi ...","This parody of first-person shooter games, mil..."
8182,s8183,TV Show,The Adventures of Figaro Pho,NaN,"Luke Jurevicius, Craig Behenna, Charlotte Haml...",Australia,NaN,2015,TV-Y7,2 Seasons,"Kids' TV, TV Comedies","Imagine your worst fears, then multiply them: ..."
